In [1]:
### インポート ###

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from datetime import datetime
import time
from selenium.webdriver.common.keys import Keys
import re
import csv
import pandas as pd
from datetime import datetime, timedelta
from dataclasses import dataclass, field
from typing import List, TypeVar, Generic
import json
import os
import logging
import functools
from pathlib import Path
import sys

In [2]:
### 定数 ###

# ログ
# logging.basicConfig(
#     level=logging.INFO,
#     format='%(asctime)s - %(levelname)s - %(message)s',
#     filename=f"C:\\keiba\\tool\\log\\{datetime.now().strftime('%Y%m%d_app.log')}",
#     encoding='utf-8',
#     force=True
# )
# 設定ファイル読み込み
config_path = Path.cwd().parent / "config.xlsx"
df_config_racecourse = pd.read_excel(config_path, sheet_name="racecourse", header=0)
df_config_style = pd.read_excel(config_path, sheet_name="style", header=0)
df_config_scrape = pd.read_excel(config_path, sheet_name="scrape", header=None, index_col=0)
df_config_netkeiba = pd.read_excel(config_path, sheet_name="netkeiba", header=None, index_col=0)
# 対象競馬場とレース
PLACE_MAP = df_config_racecourse.set_index('key')['value'].to_dict()
print(f"競馬場: {PLACE_MAP}")
# 脚質
STYLE_MAP = df_config_style.set_index('key')['value'].to_dict()
print(f"脚質: {STYLE_MAP}")
# レース番号
RACE_NUMBERS = [f"{i:02d}" for i in range(1, 13)]
# スクレイピング
PATH_CHROME_DRIVER = df_config_scrape.loc["PATH_CHROME_DRIVER"].iloc[0]
# netkeiba
LOGIN_URL = df_config_netkeiba.loc["LOGIN_URL"].iloc[0]
LOGIN_ID = df_config_netkeiba.loc["LOGIN_ID"].iloc[0]
LOGIN_PASSWORD = df_config_netkeiba.loc["LOGIN_PASSWORD"].iloc[0]
# 天気
LIST_WEATHER = ["小雨", "小雪", "晴", "雨", "曇", "雪"]
# 馬場
LIST_TRACK_CONDITION = ["重", "不", "良", "稍"]

競馬場: {35: '盛岡', 45: '川崎', 55: '佐賀', 65: '帯広(ば)', 46: '金沢', 54: '高知', 44: '大井', 47: '笠松', 50: '園田', 30: '門別', 36: '水沢', 43: '船橋', 48: '名古屋', 42: '浦和'}
脚質: {1: '逃げ', 2: '先行', 3: '差し', 4: '追込'}


In [3]:
### 関数 ###
def log_step(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # メソッド開始のログ
        logging.info(f"[開始] {func.__name__} を実行します。")
        
        try:
            # 本来のメソッド（関数）を実行
            result = func(*args, **kwargs)
            
            # メソッド終了のログ
            logging.info(f"[完了] {func.__name__} が正常に終了しました。")
            return result
            
        except Exception as e:
            # エラー発生時のログ
            logging.error(f"[失敗] {func.__name__} でエラーが発生しました: {e}")
            raise  # エラーをそのまま上に投げる
            
    return wrapper

In [4]:
#########################
# historyファイルを開く
#########################

path_history = Path.cwd().parent.parent / "data" / "history" / "history.csv"
df_history = pd.read_csv(path_history, encoding="cp932")
# 型変換
df_history["race_id"] = df_history["race_id"].astype(str)

C:\Users\ryo\AppData\Local\Temp\ipykernel_23108\2969150692.py:6: DtypeWarning: Columns (3,14,15,16,19,20,21,25,43) have mixed types. Specify dtype option on import or set low_memory=False.
  df_history = pd.read_csv(path_history, encoding="cp932")


In [5]:
df_history["race_id"]

0         202054010101
1         202054010101
2         202054010101
3         202054010101
4         202054010101
              ...     
738134    202648070312
738135    202648070312
738136    202648070312
738137    202648070312
738138    202648070312
Name: race_id, Length: 738139, dtype: object

In [6]:
#########################
# databaseファイルを開く
#########################

# ファイルを開く
path_database = Path.cwd().parent.parent / "data" / "database" / "database.csv"
df_database = pd.read_csv(path_database, encoding="cp932")
# 型変換
df_database["race_id"] = df_database["race_id"].astype(str).str.replace(r'\.0$', '', regex=True)

In [7]:
df_database["race_id"]

0       202654062702
1       202654062702
2       202654062702
3       202654062702
4       202654062702
            ...     
2846    202648070212
2847    202648070212
2848    202648070212
2849    202648070212
2850    202648070212
Name: race_id, Length: 2851, dtype: object

In [8]:
###################################################
# databaseでhistoryを更新する
###################################################

# 更新対象のレコードを切り出す
last_idx = df_history['time_index'].last_valid_index()
df_to_update = df_history.loc[last_idx + 1:].copy()
# レースID
list_race_id = df_to_update['race_id'].unique().tolist()
print(list_race_id)
df_new_data = df_database[df_database["race_id"].isin(list_race_id)]

['202654062702', '202654062703', '202654062704', '202654062705', '202654062706', '202654062707', '202654062708', '202654062709', '202654062710', '202654062711', '202635062801', '202635062802', '202635062803', '202635062804', '202635062805', '202635062806', '202635062807', '202635062808', '202635062809', '202635062810', '202635062811', '202635062812', '202665062801', '202665062802', '202665062803', '202665062804', '202665062805', '202665062806', '202665062807', '202665062808', '202665062809', '202665062810', '202665062811', '202665062812', '202646062801', '202646062802', '202646062803', '202646062804', '202646062805', '202646062806', '202646062807', '202646062808', '202646062809', '202646062810', '202646062811', '202654062801', '202654062802', '202654062803', '202654062804', '202654062805', '202654062806', '202654062808', '202654062809', '202654062810', '202654062811', '202643062801', '202643062802', '202643062803', '202643062804', '202643062805', '202643062806', '202643062807', '202643

In [9]:
# 更新すべきrace_idが存在する場合はdatabaseを参照して更新
if len(df_new_data) > 0:
    # 型変換
    df_history['race_id'] = df_history['race_id'].astype(str)
    df_history['horse_number'] = df_history['horse_number'].astype(str)
    df_new_data['race_id'] = df_new_data['race_id'].astype(str)
    df_new_data['horse_number'] = df_new_data['horse_number'].astype(str)
    df_new_data['time_index'] = pd.to_numeric(df_new_data['time_index'], errors='coerce')
    # 更新
    df_history.set_index(['race_id', 'horse_number'], inplace=True)
    df_new_data.set_index(['race_id', 'horse_number'], inplace=True)
    df_history.update(df_new_data[['time_index', 'position_1', 'position_2', 'position_3', 'position_4', 'weather_name', 'track_condition_name']])
    df_history.reset_index(inplace=True)
    # CSV出力
    df_history.to_csv(path_history, index=False, encoding="cp932")

C:\Users\ryo\AppData\Local\Temp\ipykernel_23108\1625350896.py:12: PerformanceWarning: indexing past lexsort depth may impact performance.
  df_history.update(df_new_data[['time_index', 'position_1', 'position_2', 'position_3', 'position_4', 'weather_name', 'track_condition_name']])
C:\Users\ryo\AppData\Local\Temp\ipykernel_23108\1625350896.py:12: PerformanceWarning: indexing past lexsort depth may impact performance.
  df_history.update(df_new_data[['time_index', 'position_1', 'position_2', 'position_3', 'position_4', 'weather_name', 'track_condition_name']])
C:\Users\ryo\AppData\Local\Temp\ipykernel_23108\1625350896.py:12: PerformanceWarning: indexing past lexsort depth may impact performance.
  df_history.update(df_new_data[['time_index', 'position_1', 'position_2', 'position_3', 'position_4', 'weather_name', 'track_condition_name']])
C:\Users\ryo\AppData\Local\Temp\ipykernel_23108\1625350896.py:12: PerformanceWarning: indexing past lexsort depth may impact performance.
  df_history.